# Optional RL Workshop Challenge — Equities, Bonds & Cash

**Question:** Can a tabular Q-learning agent learn a defensible dynamic allocation across SPY (equity ETF), TLT (long-duration Treasury ETF), and zero-yield cash? The task is **not** to maximize backtested return at any cost. Compare return, drawdown, trading frequency and consistency across market regimes.

**Source & limitations.** Weekly last-trading-day closing prices for 2016–2024 are derived from [m-khut/etf-analaysis](https://github.com/m-khut/etf-analaysis). They are **price observations, not verified dividend-adjusted total returns**. This is an educational toy environment with no taxes, spreads, realistic cash yield, slippage, or weight drift.

The implementation below imports the **same** `MultiAssetAgent` class embedded in the HTML workshop through Pyodide.

## 1 · The Markov decision process

State: the signs of four-week SPY and TLT momentum, plus previous discrete allocation. This creates \(2\times 2\times6=24\) states. Six available actions are cash, 100% SPY, 100% TLT, 50/50 SPY–TLT, 50% SPY/cash, and 50% TLT/cash. The Q-table contains \(24\times6=144\) estimates.

The training reward is relative to a fixed 50/50 reference (not the reported actual portfolio return):

\[
r_{t+1}^{\mathrm{train}}=w_t^\top R_{t+1}-c\,\mathrm{turnover}_t-\tfrac12(R_{SPY,t+1}+R_{TLT,t+1})-0.25\max\left(-w_t^\top R_{t+1},0\right).
\]

The reference term depends only on observed exogenous asset prices. Actual portfolio evaluation uses the **net return after transaction costs**, without subtracting the benchmark or the downside penalty.

\[
Q(s_t,a_t)\leftarrow Q(s_t,a_t)+\alpha\left[r_{t+1}+\gamma\max_a Q(s_{t+1},a)-Q(s_t,a_t)\right].
\]

Data partitions are based on the **realization date** of the return: 2016–2019 training, 2020–2022 validation, 2023–2024 final holdout. Repeating a training pass does not create independent observations.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt

root = Path.cwd()
if not (root / 'asset_agent.py').exists():
    root = root.parent
sys.path.insert(0, str(root))
from asset_agent import MultiAssetAgent, MULTI_ACTION_NAMES, greedy

data = pd.read_csv(root / 'data' / 'etf_weekly_SPY_TLT_2016_2024.csv')
data['date'] = pd.to_datetime(data['date'])
print('Observations:',len(data),'from',data.date.min().date(),'to',data.date.max().date())
data.head()

## 2 · Train the actual agent

Change \(\alpha\), \(\gamma\), exploration probability or the number of passes. All market returns, fee assumptions and reward definition remain fixed across runs. Examine not only the last reward, but the **learned policy**.

In [ ]:
cfg = dict(alpha=0.15, gamma=0.90, epsilon=0.25, budget=100, seed=17)
agent = MultiAssetAgent(data.date.dt.strftime('%Y-%m-%d').tolist(),
                        data.spy_close.tolist(), data.tlt_close.tolist(), **cfg)
agent.step_many(len(agent.train_t) * agent.budget)
print('Train / validation / final weeks:',len(agent.train_t),len(agent.val_t),len(agent.final_t))
print('Completed passes:',agent.passes,'Q-table dimensions:',len(agent.q),'×',len(agent.q[0]))
print('Greedy action counts:',{label:sum(greedy(row)==i for row in agent.q)
      for i,label in enumerate(MULTI_ACTION_NAMES)})

In [ ]:
fig,ax=plt.subplots(figsize=(10,3.5))
ax.plot(agent.history,label='Exploring-agent reward per pass')
ax.plot(agent.greedy_history,label='Greedy policy reward on training history')
ax.set(xlabel='Completed training pass',ylabel='Sum of benchmark-relative weekly rewards',
       title='Training rewards are NOT out-of-sample portfolio returns')
ax.legend();plt.show()

## 3 · Inspect what the policy learned

Choose a state with SPY momentum up, TLT momentum down and previous allocation cash. Observe six separate action values—not just one final portfolio return.

In [ ]:
state = (2*1 + 0)*6 + 0  # SPY momentum up, TLT down, previously cash
q_history = [q[state] for q in agent.q_trace]
fig,ax=plt.subplots(figsize=(10,4))
for i,label in enumerate(MULTI_ACTION_NAMES):
    ax.plot([row[i] for row in q_history],label=label)
ax.set(xlabel='Training pass',ylabel='Q-value',title=f'Actual Q-value trajectory for state {state}')
ax.legend(ncol=2,fontsize=8);plt.show()
print('Learned greedy decision:', MULTI_ACTION_NAMES[greedy(agent.q[state])])

## 4 · Risk versus return, by market regime

Validation **never** updates Q-values. Use 2020 (shock/recovery), 2021 (equity rise) and 2022 (equity/bond drawdown) as fixed chronological subperiods, not as independently sampled environments. Compare volatility, maximum drawdown, turnover and Sharpe (with cash/risk-free rate modeled as zero).

In [ ]:
v=agent.evaluate('validation')
comparison=pd.DataFrame([{
    'strategy':r['name'], 'net_return':r['totalReturn'],
    'max_drawdown':r['maxDrawdown'], 'annualized_vol':r['annualVol'],
    'sharpe_zero_cash_rate':r['sharpe'], 'allocation_changes':r['changes'],
    'worst_year':r['worstYear']} for r in v])
display(comparison.round(3))
regimes=[]
for r in v:
    for y,m in r['yearMetrics'].items():
        regimes.append({'strategy':r['name'],'year':y,
                        'return':m['totalReturn'],'max_drawdown':m['maxDrawdown']})
display(pd.DataFrame(regimes).pivot(index='year',columns='strategy',values='return').round(3))

## 5 · Optional independent check

The 2023–2024 period should be evaluated **only after choosing the model from validation**. The webpage permits one reveal in its session, but refreshing a page is not statistical independence. Even results on a formally later period are not independent once they have informed strategy redesign.

In [ ]:
# Enable once when you want the final, not during every tuning iteration:
# final = agent.evaluate('final')
# display(pd.DataFrame([{'strategy':r['name'], 'net_return':r['totalReturn'],
#                        'max_drawdown':r['maxDrawdown']} for r in final]).round(3))

### Questions for the workshop

1. When the Q-table becomes more complex, does that necessarily improve generalization?
2. Can cash become attractive because the reward penalizes absolute downside? How does using a benchmark-relative reward affect learning?
3. Did the learned allocation reduce the 2022 drawdown compared with a static 50/50 SPY–TLT strategy?
4. Are observations of 2016–2024 sufficiently independent for a meaningful claim of superior portfolio management? (No—this is still a small, single historical path.)

**Research ethics:** Historical returns and model choices have been inspected during development. Do not cite these notebook outcomes as out-of-sample evidence of investment skill.